# Step 03: Category-Stratified Train/Val/Test Split
### Iterative Multi-Label Stratification into 80% Train, 10% Val, and 10% Test
Directly extracted from `multi_class_train_rfdetr.ipynb` (Cells 1, 2, 3, 6, 9, 10, 11, 12, 13).

### Architectural Guarantees:
- **Equal Proportions**: Preserves exact class distributions across Train, Val, and Test splits.
- **Zero-Copy Architecture**: Images remain in a single shared repository (`coco_files/images/`); split folders contain only lightweight COCO JSON files and soft symlinks.
- **Step B Verification**: Validates 0-indexed contiguous category IDs, split coverage, and box boundaries.

In [ ]:
# STEP 0 - Environment Dependencies
!pip uninstall -y opencv-python opencv-contrib-python opencv-python-headless
!pip install --no-cache-dir opencv-python-headless==4.9.0.80
%pip install python-dotenv

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)


In [ ]:
# ==============================================================================
# CELL 2 - DATASET SPLIT CONFIGURATION & DIRECTORY SETUP
# Pure dataset configuration: NO model loading or training hyperparameter dependencies
# ==============================================================================

PIPELINE_NAME = "multi_class_train_rfdetr"
SAMPLE_SIZE = None              # Set integer (e.g. 1000) for test sample mode; None for full dataset
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"
RANDOM_SEED = 42

# Dataset Split Proportions (80% Train, 10% Val, 10% Test)
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO  = 0.10

# COCO Annotation Field Names
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# Repository Root & Pipeline Directories
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path(".").resolve())
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images"),
    Path("/home/jupyter/multi_class_train_rfdetr/images"),
    Path("/home/jupyter/coco_files/images"),
    Path("/home/jupyter/images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)
RAW_IMAGES_DIR = IMAGES_DIR

# Dataset Split Directories (Zero-Copy Architecture)
DATASET_DIR = os.path.join(PIPELINE_DIR, f"dataset_{MODE_TAG}")
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(DATASET_DIR, s) for s in ["train", "val", "test"]]
TRAIN_ANN, VAL_ANN, TEST_ANN = [os.path.join(d, "_annotations.coco.json") for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]]

# Logging & Checkpoint Support Directories
OUTPUT_DIR = os.path.join(PIPELINE_DIR, "runs")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
INFERENCE_OUTPUT_DIR = os.path.join(PIPELINE_DIR, "inference")
FINAL_MODEL_DIR = os.path.join(PIPELINE_DIR, "model")

logger.info("=" * 75)
logger.info(f"Step 03 Split Configuration Loaded | Mode={MODE_TAG}")
logger.info(f"   - Split Ratios:     Train={TRAIN_RATIO*100:.0f}%, Val={VALID_RATIO*100:.0f}%, Test={TEST_RATIO*100:.0f}%")
logger.info(f"   - Output Dataset:   {DATASET_DIR}")
logger.info(f"   - Input JSONs:      {INPUT_JSON_DIR.resolve()}")
logger.info(f"   - Images Pool:      {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")
logger.info("=" * 75)

In [ ]:
# CELL 3 - Dynamic Folder Creation & File Logging
for p in [DATASET_DIR, IMAGES_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

log_file_path = os.path.join(LOG_DIR, "pipeline.log")
for h in [h for h in logger.handlers if isinstance(h, (logging.FileHandler, FlushFileHandler))]:
    logger.removeHandler(h); h.close()

file_handler = FlushFileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

def tail_log(n: int = 25):
    """View latest log entries from inside Jupyter."""
    if os.path.exists(log_file_path):
        with open(log_file_path, "r", encoding="utf-8") as f:
            print("".join(f.readlines()[-n:]))

load_dotenv()
conn_str = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(conn_str) if conn_str else None
except Exception as e:
    blob_service_client = None

logger.info(f"Directories ready. Active file logger writing to: {log_file_path}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

# Check if Step 02 produced a sanitized dataset
cleaned_coco_file = REPO_ROOT / "multi_class_train_rfdetr" / "cleaned_dataset" / "_annotations.coco.json"
if cleaned_coco_file.exists() and cleaned_coco_file.stat().st_size > 0:
    logger.info(f"✓ Using sanitized dataset from Step 02: {cleaned_coco_file}")
    json_files = [cleaned_coco_file]
else:
    if not INPUT_JSON_DIR.exists():
        INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)
    json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in tqdm(json_files, desc="[CELL 4] Reading JSON files", unit="file"):
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in tqdm(raw_records, desc="[CELL 5] Parsing annotations", unit="rec"):
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 6 - Group Annotations & Apply Stratified Sampling Mode
logger.info("=" * 70)
logger.info(f"[CELL 6] Applying mode selection: {MODE_TAG.upper()}")
logger.info("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    logger.info(f"   Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    logger.info(f"   Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

logger.info(f"   - Total images in pipeline:         {len(image_urls)}")
logger.info(f"   - Total bounding boxes in pipeline: {total_selected_boxes}")

In [ ]:
# CELL 9 - Category-Stratified Train/Val/Test Split (Equal Proportions)
logger.info("=" * 70)
logger.info("[CELL 9] Performing Category-Stratified Split (Train 80%, Val 10%, Test 10%)...")
logger.info("=" * 70)

# Check if existing split annotations already exist on disk to guarantee 100% consistency
split_files_exist = all(os.path.exists(p) and os.path.getsize(p) > 100 for p in [TRAIN_ANN, VAL_ANN, TEST_ANN])

if split_files_exist:
    logger.info("=" * 70)
    logger.info(f"[DATASET CONSISTENCY] Existing Train/Val/Test splits found in {DATASET_DIR}!")
    logger.info("Reusing EXACT existing splits: Train, Val, and Test remain 100% identical and unchanged.")
    logger.info("=" * 70)
    
    with open(TRAIN_ANN, "r", encoding="utf-8") as f:
        train_coco = json.load(f)
    with open(VAL_ANN, "r", encoding="utf-8") as f:
        val_coco = json.load(f)
    with open(TEST_ANN, "r", encoding="utf-8") as f:
        test_coco = json.load(f)
        
    train_urls = [im.get("original_url", im["file_name"]) for im in train_coco["images"]]
    val_urls = [im.get("original_url", im["file_name"]) for im in val_coco["images"]]
    test_urls = [im.get("original_url", im["file_name"]) for im in test_coco["images"]]
    
    logger.info(f"   - Train split: {len(train_urls):,} images (100% preserved from existing split)")
    logger.info(f"   - Val split:   {len(val_urls):,} images (100% preserved from existing split)")
    logger.info(f"   - Test split:  {len(test_urls):,} images (100% preserved from existing split)")
else:
    logger.info("[CELL 9] Creating initial Category-Stratified Split using fixed RANDOM_SEED (42)...")
    valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
    random.seed(RANDOM_SEED)

    cat_to_valid_imgs: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in valid_urls:
        p_cat = image_primary_cat.get(u, AUTO_CATEGORIES[0])
        cat_to_valid_imgs[p_cat].append(u)

    train_urls, val_urls, test_urls = [], [], []
    for cat, imgs in cat_to_valid_imgs.items():
        random.shuffle(imgs)
        n = len(imgs)
        n_tr = int(n * TRAIN_RATIO)
        n_va = int(n * VALID_RATIO)
        train_urls.extend(imgs[:n_tr])
        val_urls.extend(imgs[n_tr:n_tr + n_va])
        test_urls.extend(imgs[n_tr + n_va:])

def count_category_boxes(urls: List[str]) -> Dict[str, int]:
    counts = {cat: 0 for cat in AUTO_CATEGORIES}
    for u in urls:
        for a in filtered_annotations.get(u, []):
            c = a["category_name"]
            counts[c] = counts.get(c, 0) + 1
    return counts

train_cat_boxes = count_category_boxes(train_urls)
val_cat_boxes = count_category_boxes(val_urls)
test_cat_boxes = count_category_boxes(test_urls)
total_cat_boxes = {cat: train_cat_boxes[cat] + val_cat_boxes[cat] + test_cat_boxes[cat] for cat in AUTO_CATEGORIES}

proportions_rows = []
for cat in AUTO_CATEGORIES:
    tot = max(1, total_cat_boxes[cat])
    proportions_rows.append({
        "Category": cat,
        "Total Boxes": total_cat_boxes[cat],
        "Train Count": train_cat_boxes[cat],
        "Train %": f"{(train_cat_boxes[cat]/tot)*100:.1f}%",
        "Val Count": val_cat_boxes[cat],
        "Val %": f"{(val_cat_boxes[cat]/tot)*100:.1f}%",
        "Test Count": test_cat_boxes[cat],
        "Test %": f"{(test_cat_boxes[cat]/tot)*100:.1f}%",
    })

proportions_df = pd.DataFrame(proportions_rows)
logger.info("Category Proportions Across Splits:")
logger.info("\n" + proportions_df.to_string(index=False))
logger.info(f"Total Images: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}")

In [ ]:
# CELL 10 - Prepare Split Annotation Directories (Zero-Copy Architecture)
logger.info("=" * 70)
logger.info("[CELL 10] Preparing split annotation directories (Zero-Copy Architecture)...")
logger.info("=" * 70)

for split_name in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_name, exist_ok=True)
    images_link = os.path.join(split_name, "images")
    if not os.path.exists(images_link):
        try:
            os.symlink(os.path.abspath(IMAGES_DIR), images_link)
        except OSError:
            pass

logger.info(f"   - Shared images directory:  {IMAGES_DIR} ({len(os.listdir(IMAGES_DIR))} files)")
logger.info(f"   - Train split annotations:  {TRAIN_DIR} ({len(train_urls)} images)")
logger.info(f"   - Val split annotations:    {VAL_DIR} ({len(val_urls)} images)")
logger.info(f"   - Test split annotations:   {TEST_DIR} ({len(test_urls)} images)")
logger.info("Split directories ready with zero redundant image copying.")

In [ ]:
# CELL 11 - Build Multi-Class COCO Annotations
logger.info("=" * 70)
logger.info(f"[CELL 11] Building COCO annotations for {NUM_CLASSES} discovered categories...")
logger.info("=" * 70)

categories_def = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]

def build_coco_for_split(urls: List[str], split_dir: str, ann_path: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": int(cid),
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Multi-Class Dataset ({NUM_CLASSES} classes)", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    logger.info(f"   - Saved: {ann_path} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

# Only build if split files do not already exist (preserves existing dataset splits 100% intact)
split_files_exist = all(os.path.exists(p) and os.path.getsize(p) > 100 for p in [TRAIN_ANN, VAL_ANN, TEST_ANN])

if split_files_exist:
    logger.info("=" * 70)
    logger.info("[DATASET INTEGRITY] Train, Val, and Test split annotations already exist.")
    logger.info("Preserving existing split files 100% untouched to ensure exact consistency across model versions.")
    logger.info(f"   - Train: {TRAIN_ANN}")
    logger.info(f"   - Val:   {VAL_ANN}")
    logger.info(f"   - Test:  {TEST_ANN}")
    logger.info("=" * 70)
else:
    build_coco_for_split(train_urls, TRAIN_DIR, TRAIN_ANN)
    build_coco_for_split(val_urls, VAL_DIR, VAL_ANN)
    build_coco_for_split(test_urls, TEST_DIR, TEST_ANN)

In [ ]:
# CELL 12 - Dataset Inspection & Step B Label/Category Alignment Verification
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

# ==========================================================================
# STEP B CHECKS: Comprehensive Label & Category Alignment Diagnostic Suite
# ==========================================================================
from collections import defaultdict, Counter
def verify_label_and_category_alignment(train_path: str, val_path: str, test_path: str) -> dict:
    """Rigorous Step B diagnostic check verifying category indexing, split coverage, and box coordinates."""
    logger.info("=" * 75)
    logger.info("[STEP B CHECK] Executing Label & Category Alignment Verification Suite...")
    logger.info("=" * 75)
    
    issues = []
    splits = {"train": train_path, "val": val_path, "test": test_path}
    loaded = {}
    for sname, spath in splits.items():
        with open(spath, "r", encoding="utf-8") as f:
            loaded[sname] = json.load(f)

    # Check 1: Contiguous 0-indexing in categories
    train_cats = loaded["train"].get("categories", [])
    train_ids = sorted([c["id"] for c in train_cats])
    expected_ids = list(range(len(train_cats)))
    if train_ids != expected_ids:
        msg = f"Category IDs are not contiguous 0-indexed! Expected {expected_ids[:5]}..., got {train_ids[:5]}..."
        issues.append(msg)
        logger.error(f"   [FAIL] {msg}")
    else:
        logger.info(f"   [PASS] Contiguous 0-indexing verified: IDs {min(train_ids)} to {max(train_ids)} ({len(train_ids)} classes).")

    # Check 2: Cross-split category consistency
    train_cat_dict = {c["id"]: c["name"] for c in train_cats}
    for sname in ["val", "test"]:
        s_ids = set(c["id"] for c in loaded[sname].get("categories", []))
        diff = s_ids - set(train_ids)
        if diff:
            msg = f"{sname.capitalize()} contains category IDs not present in Train: {diff}"
            issues.append(msg)
            logger.error(f"   [FAIL] {msg}")
        else:
            logger.info(f"   [PASS] All {sname} categories align exactly with train categories.")

    # Check 3: Bounding box validity and class counts
    ann_counts = {s: defaultdict(int) for s in splits}
    degenerate_boxes = 0
    for sname, data in loaded.items():
        imgs = {im["id"]: im for im in data.get("images", [])}
        for ann in data.get("annotations", []):
            cid = ann.get("category_id")
            ann_counts[sname][cid] += 1
            bbox = ann.get("bbox", [])
            if len(bbox) != 4 or bbox[2] <= 0 or bbox[3] <= 0:
                degenerate_boxes += 1
                continue
            x, y, w, h = bbox
            im_meta = imgs.get(ann.get("image_id"), {})
            im_w, im_h = im_meta.get("width", 1920), im_meta.get("height", 1080)
            if x < 0 or y < 0 or (x + w) > im_w * 1.05 or (y + h) > im_h * 1.05:
                degenerate_boxes += 1

    logger.info("-" * 75)
    logger.info(f"{'ID':<5} {'Category Name':<25} {'Train Boxes':>12} {'Val Boxes':>11} {'Test Boxes':>11} {'Status':>8}")
    logger.info("-" * 75)
    for cid in train_ids:
        cname = train_cat_dict.get(cid, "Unknown")
        tr_c = ann_counts["train"][cid]
        va_c = ann_counts["val"][cid]
        te_c = ann_counts["test"][cid]
        status = "OK"
        if va_c == 0:
            status = "NO VAL!"
            issues.append(f"Category '{cname}' (ID {cid}) has 0 instances in validation split!")
        elif tr_c < 5:
            status = "LOW DATA"
            issues.append(f"Category '{cname}' (ID {cid}) has only {tr_c} train instances (<5).")
        logger.info(f"{cid:<5} {cname[:24]:<25} {tr_c:>12} {va_c:>11} {te_c:>11} {status:>8}")
    logger.info("-" * 75)

    if degenerate_boxes > 0:
        msg = f"Detected {degenerate_boxes} degenerate or out-of-bounds bounding boxes across dataset splits!"
        issues.append(msg)
        logger.warning(f"   [WARN] {msg}")
    else:
        logger.info("   [PASS] All bounding boxes verified: valid positive width/height within image limits.")

    if not issues:
        logger.info("[STEP B CHECK PASSED] Dataset labels, indexing, and splits are fully consistent.")
    else:
        logger.warning(f"[STEP B CHECK NOTICE] {len(issues)} item(s) to be aware of. Validation loss may be affected by class distribution.")
    logger.info("=" * 75)
    return {"issues": issues, "ann_counts": ann_counts}

step_b_audit = verify_label_and_category_alignment(TRAIN_ANN, VAL_ANN, TEST_ANN)

In [ ]:
# CELL 13 - Print Sample Images per Tag (In-Memory Display - No Files Saved)
from PIL import Image, ImageDraw
from IPython.display import display
from collections import defaultdict
from pathlib import Path
import os, json

# High-contrast color palette for distinct multi-class boxes
SAMPLE_COLORS = [
    (230, 25, 75),   # Red
    (60, 180, 75),   # Green
    (255, 225, 25),  # Yellow
    (67, 99, 216),   # Blue
    (245, 130, 49),  # Orange
    (145, 30, 180),  # Purple
    (66, 212, 244),  # Cyan
    (240, 50, 230),  # Magenta
    (191, 239, 69),  # Lime
    (250, 190, 212), # Pink
    (70, 153, 144),  # Teal
    (220, 190, 255)  # Lavender
]

def _safe_cat_key(val, default=0):
    """Normalize any category_id (int, string, list, dict) into a hashable identifier."""
    if isinstance(val, (list, tuple)):
        val = val[0] if len(val) > 0 else default
    elif isinstance(val, dict):
        val = val.get("id", val.get("category_id", val.get("name", default)))
    if val is None:
        return default
    try:
        return int(val)
    except (ValueError, TypeError):
        return str(val)

def print_sample_images_per_tag(annotation_source=None, image_dir=None, max_display_width: int = 700):
    """
    Selects sample images covering each discovered category tag, annotates
    bounding boxes with colored badges in memory, and prints/displays them
    inline directly in the notebook without saving any files to disk.
    
    Accepts:
    - annotation_source: Path or string to COCO JSON file, OR loaded dict/list
    - image_dir: Path or directory where images are cached
    """
    print("=" * 75)
    print("[CELL 13] Printing Sample Images with Category Tags (No Disk Saving)")
    print("=" * 75)

    if image_dir is None:
        image_dir = IMAGES_DIR if "IMAGES_DIR" in globals() else "images"
    image_dir = str(image_dir)

    categories = {}
    img_id_map = {}
    cat_to_images = defaultdict(list)
    img_to_anns = defaultdict(list)

    # Load annotations whether passed as path, dict, or list of records
    data = None
    if isinstance(annotation_source, (str, Path)) and os.path.exists(str(annotation_source)):
        with open(str(annotation_source), "r", encoding="utf-8") as f:
            data = json.load(f)
    elif isinstance(annotation_source, (dict, list)):
        data = annotation_source
    elif "image_annotations" in globals() and isinstance(globals()["image_annotations"], dict):
        # Fallback to parsed in-memory image annotations
        data = None

    if isinstance(data, dict):
        for c in data.get("categories", []):
            if isinstance(c, dict):
                cid = _safe_cat_key(c.get("id", 0))
                categories[cid] = str(c.get("name", cid))
        for im in data.get("images", []):
            if isinstance(im, dict):
                img_id_map[im.get("id")] = im
        for ann in data.get("annotations", []):
            if isinstance(ann, dict):
                cid = _safe_cat_key(ann.get("category_id", 0))
                iid = ann.get("image_id")
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(ann)
    elif isinstance(data, list):
        for idx, item in enumerate(data):
            if isinstance(item, dict):
                iid = item.get(globals().get("IMAGE_FIELD", "image_id"), idx)
                raw_cid = item.get("category_id", item.get(globals().get("CATEGORY_FIELD", "category_id"), 0))
                cid = _safe_cat_key(raw_cid)
                
                raw_cat_name = item.get("category_name", item.get(globals().get("CATEGORY_FIELD", "category_id"), f"class_{cid}"))
                if isinstance(raw_cat_name, (list, tuple)):
                    cat_name = str(raw_cat_name[0]) if len(raw_cat_name) > 0 else f"class_{cid}"
                elif isinstance(raw_cat_name, dict):
                    cat_name = str(raw_cat_name.get("name", raw_cat_name.get("id", f"class_{cid}")))
                else:
                    cat_name = str(raw_cat_name)
                    
                categories[cid] = cat_name
                fname = Path(str(item.get("image_url", item.get("coco_url", f"{iid}.jpg")))).name
                img_id_map[iid] = {"id": iid, "file_name": fname}
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(item)
    elif "image_annotations" in globals():
        in_mem = globals()["image_annotations"]
        in_cat_to_id = globals().get("cat_to_id", {})
        categories = {_safe_cat_key(idx): cat for cat, idx in in_cat_to_id.items()}
        for iid, anns in in_mem.items():
            fname = Path(str(iid)).name if "/" in str(iid) else f"{iid}.jpg"
            img_id_map[iid] = {"id": iid, "file_name": fname}
            for a in anns:
                cid = _safe_cat_key(a.get("category_id", 0))
                cat_to_images[cid].append(iid)
                img_to_anns[iid].append(a)

    if not categories and "AUTO_CATEGORIES" in globals():
        categories = {_safe_cat_key(idx): cat for idx, cat in enumerate(globals()["AUTO_CATEGORIES"])}

    # Match with existing local image files across all candidate directories
    search_dirs = [
        str(image_dir),
        str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
        str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
        str(Path("coco_files/images")),
        str(Path("images")),
        "/home/jupyter/multi_class_train_rfdetr/images",
        "/home/jupyter/coco_files/images",
        "/home/jupyter/images"
    ]
    search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

    existing_cat_images = {}
    for cid, iids in cat_to_images.items():
        valid_pairs = []
        for iid in iids:
            im_meta = img_id_map.get(iid, {})
            fname = im_meta.get("file_name", f"{iid}.jpg") if isinstance(im_meta, dict) else f"{iid}.jpg"
            
            candidate_names = [
                fname,
                Path(fname).name if fname else None,
                f"{iid}.jpg",
                f"{Path(str(iid)).name}",
                f"{Path(str(iid)).stem}.jpg"
            ]
            candidate_names = [n for n in candidate_names if n]

            found_path = None
            for d in search_dirs:
                for name in candidate_names:
                    p = os.path.join(d, name)
                    if os.path.exists(p) and os.path.getsize(p) > 0:
                        found_path = p
                        break
                if found_path:
                    break

            if found_path:
                valid_pairs.append((iid, found_path))
        if valid_pairs:
            existing_cat_images[cid] = valid_pairs

    if not existing_cat_images:
        print("[WARN] No matching local images found on disk to display yet.")
        print(f"       Searched image directories:")
        for d in search_dirs:
            count = len(os.listdir(d)) if os.path.exists(d) else 0
            print(f"         - {os.path.abspath(d)} ({count} files)")
        print("       (Run '00_download_coco_files.ipynb' first to download images into the shared pool)")
        return

    # Display an annotated sample for every distinct category tag (sort safely)
    sorted_cat_keys = sorted(existing_cat_images.keys(), key=lambda k: (0, k) if isinstance(k, int) else (1, str(k)))
    for cid in sorted_cat_keys:
        cat_name = categories.get(cid, str(cid))
        sample_img_id, img_path = existing_cat_images[cid][0]
        fname = os.path.basename(img_path)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARN] Could not open image {img_path}: {e}")
            continue

        im_w, im_h = img.size
        draw = ImageDraw.Draw(img)
        anns = img_to_anns.get(sample_img_id, [])

        cat_counts = defaultdict(int)
        for ann in anns:
            if not isinstance(ann, dict):
                continue
            raw_bbox = ann.get("bbox", [])
            if len(raw_bbox) != 4:
                continue
            try:
                x, y, w, h = [float(v) for v in raw_bbox]
            except (ValueError, TypeError):
                continue
            if w <= 0 or h <= 0:
                continue

            ann_cid = _safe_cat_key(ann.get("category_id", 0))
            ann_cname = str(categories.get(ann_cid, ann_cid))
            cat_counts[ann_cname] += 1
            color_idx = ann_cid if isinstance(ann_cid, int) else (hash(str(ann_cid)) & 0x7FFFFFFF)
            color = SAMPLE_COLORS[color_idx % len(SAMPLE_COLORS)]

            # Clamp box to valid dimensions
            x0 = max(0.0, min(float(im_w - 1), x))
            y0 = max(0.0, min(float(im_h - 1), y))
            x1 = max(0.0, min(float(im_w - 1), x + w))
            y1 = max(0.0, min(float(im_h - 1), y + h))

            # Draw bounding box
            draw.rectangle([x0, y0, x1, y1], outline=color, width=4)

            # Draw colored label badge with tag name
            label_text = f" {ann_cname} "
            if hasattr(draw, "textbbox"):
                tb = draw.textbbox((x0, max(0.0, y0 - 18.0)), label_text)
                draw.rectangle(tb, fill=color)
                draw.text((x0, max(0.0, y0 - 18.0)), label_text, fill=(255, 255, 255))
            else:
                draw.text((x0 + 2, max(0.0, y0 - 14.0)), label_text, fill=color)

        summary_str = ", ".join(f"'{k}': {v}" for k, v in sorted(cat_counts.items()))
        print(f"\n>>> Target Tag: [{cat_name}] | File: {fname} ({im_w}x{im_h}) | Boxes: {len(anns)} ({summary_str})")

        # Scale in-memory copy for clean notebook view if wide
        if im_w > max_display_width:
            scale = max_display_width / im_w
            display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
        else:
            display_img = img

        # Directly print/display in notebook (Zero files saved to disk)
        display(display_img)

    print("\n" + "=" * 75)
    print("All category tag samples displayed inline successfully.")
    print("=" * 75)

# Auto-discover first available annotation file or in-memory annotations
sample_ann_candidates = sorted(list(INPUT_JSON_DIR.glob("*.json"))) if "INPUT_JSON_DIR" in globals() else []
target_ann_file = sample_ann_candidates[0] if sample_ann_candidates else None

if target_ann_file and target_ann_file.exists():
    logger.info(f"Using '{target_ann_file.name}' for inline sample visualization...")
    print_sample_images_per_tag(str(target_ann_file), str(IMAGES_DIR))
else:
    # Use in-memory parsed annotations
    print_sample_images_per_tag(None, str(IMAGES_DIR))